In [ ]:
!pip install transformers accelerate bitsandbytes sentencepiece --quiet

In [ ]:
!pip install einops

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm

In [ ]:

questions = {
    "A_famous_facts": [
        "Who is the current president of the United States?",
        "Who wrote the play 'Hamlet'?",
        "What is the capital of France?",
        "What is the speed of light?",
        "Who was Albert Einstein?",
        "What year did World War II end?",
        "What is the largest ocean on Earth?",
        "What is photosynthesis?",
        "Who painted the Mona Lisa?",
        "What is gravity?",
    ],
    "B_fictional_worlds": [
        "Who is Harry Potter’s best friend?",
        "What is the One Ring in Lord of the Rings?",
        "What is the main spaceship in Star Wars?",
        "Who is Sherlock Holmes’ partner?",
        "What is the force used by Jedi?",
        "What is the name of the wizarding school in Harry Potter?",
        "Who is the main villain in Lord of the Rings?",
        "What is the Stark family known for?",
        "What is the Iron Throne?",
        "Who is Frodo Baggins?",
    ],
    "C_technical": [
        "What is backpropagation?",
        "What is a qubit?",
        "What is Fourier transform used for?",
        "What is gradient descent?",
        "What is the purpose of a compiler?",
        "What is a hash function?",
        "What is a neural network?",
        "What is entropy in information theory?",
        "What is the difference between RAM and ROM?",
        "What is a TCP handshake?",
    ],
    "D_biography": [
        "Who is John Smith?",
        "Describe a typical person named Emily Johnson.",
        "Who is Michael Chen?",
        "What is the life story of Sarah Thompson?",
        "What does David Patel do for a living?",
        "Invent a biography for Anna Lopez.",
        "Tell me about James Walker’s childhood.",
        "Who is Priya Sharma?",
        "What is the career of Daniel Brooks?",
        "Who is Rebecca Lee?",
    ],
    "E_hybrid": [
        "Explain how gravity works in the Harry Potter universe.",
        "Compare the US president with the King of Gondor.",
        "How would Einstein explain the Force?",
        "Describe TCP handshake using Harry Potter characters.",
        "Explain quantum computing as if Sherlock Holmes solved it.",
        "What if the Mona Lisa were painted in Middle-Earth?",
        "Describe photosynthesis on a spaceship.",
        "Explain World War II using Star Wars terminology.",
        "What if Frodo invented backpropagation?",
        "Compare real-world gravity with fictional magic.",
    ],
    "F_counterfactual": [
        "Who was the second woman to walk on the Moon in 1972?",
        "What country elected Isaac Newton as president?",
        "In which year did the Roman Empire land on Mars?",
        "Who discovered penicillin in the year 2000?",
        "When did Japan join the African Union?",
        "Which US president invented the smartphone?",
        "What treaty ended World War III?",
        "What college did Cleopatra attend?",
        "Who invented electricity in ancient Greece?",
        "Which European country was founded in 2025?",
    ],
    "G_domain_conflict": [
        "Based on the context: 'Sherlock Holmes is a NASA engineer,' explain his job role.",
        "Given: 'The capital of France is Lyon,' summarize the geography of France.",
        "If the One Ring is described as a scientific instrument, explain its purpose.",
        "According to the context: 'Albert Einstein was a medieval king,' describe his achievements.",
        "If 'Harry Potter is a physicist,' explain his research work.",
        "Given the context: 'Photosynthesis occurs on the Moon,' describe the process.",
        "If the Mona Lisa is a robot, explain how it functions.",
        "Given: 'The TCP handshake is a type of duel,' explain the steps.",
        "If Frodo is a software engineer, describe his contributions.",
        "According to the context: 'World War II happened in 2100,' summarize the main events.",
    ],
    "H_ambiguous": [
        "Who is Alex Turner?",
        "Describe the origin of the Crimson Artifact.",
        "Explain the significance of the Omega Protocol.",
        "Tell me about the Solaris Event.",
        "What happened during the Riverfall Incident?",
        "What is the purpose of the Arclight Project?",
        "Describe a person named Jordan.",
        "What is the Helios System used for?",
        "Who is the creator of the Evergreen Method?",
        "What is the Shadow Network known for?",
    ],
}




In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

In [ ]:
with open("qwen_dataset_from_excel.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Load GPT-2
model_name = "gpt2"  # or gpt2-medium for larger
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Helper to generate text
def generate_context(prompt, max_new_tokens=50):
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        inputs,
        max_new_tokens=max_new_tokens,  # <-- use max_new_tokens instead of max_length
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # remove the prompt from the generated text
    context = text[len(prompt):].strip().split("\n")[0]  # take first line
    return context
context_dataset = {}
for domain, qa_list in data.items():
    context_dataset[domain] = []
    for qa in qa_list:
        question = qa["question"]
        answer = qa["answer"]
        # aligned context prompt
        aligned_prompt = f"Question: {question}\nAnswer: {answer}\nTask: Write a concise factual context for this question: "
        aligned_context = generate_context(aligned_prompt)
        # conflicting context prompt
        conflicting_prompt = f"Question: {question}\nAnswer: {answer}\nTask: Write a concise plausible but incorrect context for this question: "
        conflicting_context = generate_context(conflicting_prompt)

        context_dataset[domain].append({
            "question": question,
            "answer": answer,
            "aligned_context": aligned_context,
            "conflicting_context": conflicting_context
        })

# Save context-enhanced dataset
with open("qwen_context_dataset_gpt2.json", "w", encoding="utf-8") as f:
    json.dump(context_dataset, f, indent=4, ensure_ascii=False)

print("✅ Context dataset saved as qwen_context_dataset_gpt2.json")


✅ Context dataset saved as qwen_context_dataset_gpt2.json


In [ ]:


# Load dataset
with open("unified_data.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Function to get full deterministic answer
def get_full_prior_answer(question, max_new_tokens=50):
    prompt = f"Q: {question}\nA: Answer only in one or two concise lines, no extra explanation:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # deterministic
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from the generated text
    text = text.replace(prompt, "").strip()
    # Keep first meaningful line (optional)
    first_line = text.split("\n")[0].strip()
    return first_line

# Run prior probe for all questions in the dataset
prior_probe_results = {}

for domain, qa_list in dataset.items():
    prior_probe_results[domain] = []
    print(f"Processing domain: {domain} ({len(qa_list)} questions)")
    for qa in qa_list:
        question = qa["question"]
        answer = qa["answer"]
        # Generate prior answer
        prior_answer = get_full_prior_answer(question)
        prior_probe_results[domain].append({
            "question": question,
            "answer": answer,
            "prior_answer": prior_answer
        })

# Save results to JSON
with open("full_prior_probe_results.json", "w", encoding="utf-8") as f:
    json.dump(prior_probe_results, f, indent=4, ensure_ascii=False)

print("Full prior probe pass complete. Results saved in full_prior_probe_results.json")


Processing domain: A_famous_facts (10 questions)
Processing domain: B_fictional_worlds (10 questions)
Processing domain: C_technical (10 questions)
Processing domain: D_biography (10 questions)
Processing domain: E_hybrid (10 questions)
Processing domain: F_counterfactual (5 questions)
Processing domain: G_domain_conflict (5 questions)
Processing domain: H_ambiguous (5 questions)
Full prior probe pass complete. Results saved in full_prior_probe_results.json


In [ ]:
def generate_with_context(context, question, max_new_tokens=100):
    """
    Generates the model's answer given a context and a question.
    Keeps deterministic output and returns first meaningful line.
    """
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer only in one or two concise lines:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # deterministic
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    text = text.replace(prompt, "").strip()
    first_line = text.split("\n")[0].strip()
    return first_line


In [ ]:
# Load datasets
with open("unified_data.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

with open("full_prior_probe_results.json", "r", encoding="utf-8") as f:
    prior_results = json.load(f)

# Store outputs
context_pass_results = {}

for domain, qa_list in prior_results.items():
    context_pass_results[domain] = []
    print(f"Processing domain: {domain}")
    for qa in qa_list:
        question = qa["question"]
        prior_answer = qa["prior_answer"]

        # Fetch contexts from dataset
        aligned_context = next(item for item in dataset[domain] if item["question"] == question)["aligned_context"]
        conflicting_context = next(item for item in dataset[domain] if item["question"] == question)["conflicting_context"]

        # Generate outputs
        aligned_output = generate_with_context(aligned_context, question)
        conflicting_output = generate_with_context(conflicting_context, question)

        # Store everything
        context_pass_results[domain].append({
            "question": question,
            "prior_answer": prior_answer,
            "aligned_context": aligned_context,
            "aligned_output": aligned_output,
            "conflicting_context": conflicting_context,
            "conflicting_output": conflicting_output
        })

# Save results
with open("context_pass_results.json", "w", encoding="utf-8") as f:
    json.dump(context_pass_results, f, indent=4, ensure_ascii=False)

print("Context pass complete. Results saved in context_pass_results.json")


Processing domain: A_famous_facts
Processing domain: B_fictional_worlds
Processing domain: C_technical
Processing domain: D_biography
Processing domain: E_hybrid
Processing domain: F_counterfactual
Processing domain: G_domain_conflict
Processing domain: H_ambiguous
Context pass complete. Results saved in context_pass_results.json


In [ ]:
import torch
MODEL_NAME = "Qwen/Qwen2.5-7B"  # BASE model, not instruct

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True
)

def call_llm(prompt, max_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=0,
        eos_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [ ]:
def generate_prior_answer(question):
    prompt = f"""Answer the following question concisely and directly.Do not include any metadata or unecessary information.

Question: {question}

Answer:"""
    return call_llm(prompt)


In [ ]:
def generate_with_context(context, question):
    prompt = f"""
You must answer the question using ONLY the information given in the context.
The answer may be one or more sentences, depending on what the question needs.
Follow these strict rules:
- Do NOT include disclaimers or metadata.
- Produce ONLY the final answer.

Context: {context}

Question: {question}

Answer:""".strip()

    return call_llm(prompt)


In [ ]:
import json

# Load unified dataset
with open("unified_data.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

results = {}

for domain, items in dataset.items():
    results[domain] = []
    print(f"Processing {domain}...")

    for item in items:
        question = item["question"]
        aligned_context = item["aligned_context"]
        conflicting_context = item["conflicting_context"]

        # 1. Generate prior answer (no context)
        prior_answer = generate_prior_answer(question)

        # 2. Generate aligned-context answer
        aligned_output = generate_with_context(aligned_context, question)

        # 3. Generate conflicting-context answer
        conflicting_output = generate_with_context(conflicting_context, question)

        results[domain].append({
            "question": question,
            "prior_answer": prior_answer,
            "aligned_context": aligned_context,
            "aligned_output": aligned_output,
            "conflicting_context": conflicting_context,
            "conflicting_output": conflicting_output
        })

# Save
with open("context_pass_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("Done.")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Processing A_famous_facts...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

Processing B_fictional_worlds...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

...

Processing G_domain_conflict...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

Processing H_ambiguous...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

Done.


In [ ]:
!pip install -q sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer, util
import json

# Load results from context pass
with open("context_pass_results.json", "r", encoding="utf-8") as f:
    context_data = json.load(f)

# Load sentence embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_cf(context, output):
    """Compute Context Fidelity score (cosine similarity between output and context)"""
    context_emb = embed_model.encode(context, convert_to_tensor=True)
    output_emb = embed_model.encode(output, convert_to_tensor=True)
    sim = util.cos_sim(output_emb, context_emb)  # cosine similarity
    return sim.item()  # return as float

# Compute CF for all questions
for domain, qa_list in context_data.items():
    for qa in qa_list:
        aligned_output = qa["aligned_output"]
        aligned_context = qa["aligned_context"]
        cf_score = compute_cf(aligned_context, aligned_output)
        qa["CF_score"] = cf_score  # store CF

# Save updated results
with open("context_pass_results_with_CF.json", "w", encoding="utf-8") as f:
    json.dump(context_data, f, indent=4, ensure_ascii=False)

print(" Context Fidelity (CF) computed and saved in context_pass_results_with_CF.json")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Context Fidelity (CF) computed and saved in context_pass_results_with_CF.json


In [ ]:
import json
import nltk
import difflib
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt')

# -----------------------------
# Utility functions
# -----------------------------

def tokenize(text):
    return nltk.word_tokenize(text.lower())

def jaccard(a, b):
    set_a, set_b = set(a), set(b)
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)

def normalized_edit_distance(a, b):
    seq = difflib.SequenceMatcher(None, a, b)
    return seq.ratio()  # already normalized 0–1

# -----------------------------
# Metric Computations
# -----------------------------

def compute_context_fidelity(output, context):
    """
    CF = fraction of tokens in output that appear in context
    """
    o_tokens = tokenize(output)
    c_tokens = tokenize(context)

    if not o_tokens:
        return 0.0

    overlap = sum(1 for t in o_tokens if t in c_tokens)
    return overlap / len(o_tokens)


def compute_prior_dominance(output, prior_answer):
    """
    PD = fraction of output tokens that come from prior answer
    """
    o_tokens = tokenize(output)
    p_tokens = tokenize(prior_answer)

    if not o_tokens:
        return 0.0

    overlap = sum(1 for t in o_tokens if t in p_tokens)
    return overlap / len(o_tokens)


def compute_reconstruction_score(output, context, prior):
    """
    RS = 0.5 * edit_similarity + 0.5 * BLEU
    Measures how well output "reconstructs" the intended meaning.
    """

    # edit similarity relative to context and prior
    edit_c = normalized_edit_distance(output, context)
    edit_p = normalized_edit_distance(output, prior)
    edit_score = max(edit_c, edit_p)  # pick whichever is closer

    # BLEU similarity
    cc = SmoothingFunction()
    bleu_c = sentence_bleu([tokenize(context)], tokenize(output), smoothing_function=cc.method1)
    bleu_p = sentence_bleu([tokenize(prior)], tokenize(output), smoothing_function=cc.method1)
    bleu_score = max(bleu_c, bleu_p)

    return 0.5 * edit_score + 0.5 * bleu_score


def compute_topk_overlap(prior_topk, context_output):
    """
    IoU between context-grounded predictions & prior top-k strings
    """
    prior_items = set([item["answer"].lower() for item in prior_topk])
    out_tokens = set(tokenize(context_output))

    if not prior_items and not out_tokens:
        return 1.0

    return jaccard(prior_items, out_tokens)


def compute_logprob_delta(aligned_lp, conflicting_lp):
    """
    ΔLP = logprob(aligned) – logprob(conflicting)
    """
    if aligned_lp is None or conflicting_lp is None:
        return None
    return aligned_lp - conflicting_lp


# -----------------------------
# Apply metrics to JSON
# -----------------------------

def compute_all_metrics(input_file, output_file="final_evaluation_metrics.json"):
    with open(input_file, "r") as f:
        data = json.load(f)

    results = {}

    for category, items in data.items():
        results[category] = []

        for item in items:
            question = item["question"]
            #answer = item["answer"]

            # model outputs
            prior_output = item["prior_answer"]
            aligned_output = item["aligned_output"]
            conflicting_output = item["conflicting_output"]

            # contexts
            aligned_ctx = item["aligned_context"]
            conflicting_ctx = item["conflicting_context"]

            # prior data
            prior_topk = item["top5_prior"]
            aligned_lp = item.get("aligned_logprob", None)
            conflicting_lp = item.get("conflicting_logprob", None)

            # ---------- Metrics ----------
            CF = compute_context_fidelity(aligned_output, aligned_ctx)
            PD = compute_prior_dominance(aligned_output, prior_output)
            RS = compute_reconstruction_score(aligned_output, aligned_ctx, prior_output)
            TOPK = compute_topk_overlap(prior_topk, aligned_output)
            DLP = compute_logprob_delta(aligned_lp, conflicting_lp)

            results[category].append({
                "question": question,


                "prior_output": prior_output,
                "aligned_output": aligned_output,
                "conflicting_output": conflicting_output,

                # Metrics
                "Context_Fidelity": CF,
                "Prior_Dominance": PD,
                "Reconstruction_Score": RS,
                "TopK_Overlap": TOPK,
                "Delta_LogProb": DLP
            })

    with open(output_file, "w") as f:
        json.dump(results, f, indent=4)

    print("✅ Final metrics saved to", output_file)


# -----------------------------
# RUN IT
# -----------------------------
compute_all_metrics("context_pass_results.json")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


KeyError: 'top5_prior'

In [ ]:
!pip install Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.6 MB/s eta 0:00:00


In [ ]:
import json
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import Levenshtein

# -----------------------------
# Helpers
# -----------------------------

def normalize(x):
    return x.lower().strip()

def context_fidelity(output, context):
    """How much output matches the context."""
    out = normalize(output)
    ctx = normalize(context)
    overlap_words = set(out.split()) & set(ctx.split())
    return len(overlap_words) / (len(set(out.split())) + 1e-8)

def prior_dominance(output, prior):
    """How much output matches the prior answer."""
    out = normalize(output)
    pr = normalize(prior)
    overlap_words = set(out.split()) & set(pr.split())
    return len(overlap_words) / (len(set(out.split())) + 1e-8)

def reconstruction_score(output, context, prior):
    """RS = mix(edit distance + BLEU)"""
    out = normalize(output)
    ctx = normalize(context)
    pr = normalize(prior)

    # Edit distances (smaller = better)
    ed_ctx = Levenshtein.distance(out, ctx)
    ed_prior = Levenshtein.distance(out, pr)

    ed_component = 1 / (1 + min(ed_ctx, ed_prior))

    # BLEU similarity (bigger = better)
    smoothie = SmoothingFunction().method1
    bleu_ctx = sentence_bleu([ctx.split()], out.split(), smoothing_function=smoothie)
    bleu_prior = sentence_bleu([pr.split()], out.split(), smoothing_function=smoothie)

    bleu_component = max(bleu_ctx, bleu_prior)

    return 0.5 * (ed_component + bleu_component)

def topk_overlap(output, prior):
    out_words = set(normalize(output).split())
    pr_words = set(normalize(prior).split())
    inter = len(out_words & pr_words)
    union = len(out_words | pr_words) + 1e-8
    return inter / union

def logprob_delta_stub():
    """You don’t have logprobs, so always return None."""
    return None


# -----------------------------
# MAIN METRIC COMPUTATION
# -----------------------------

def compute_all_metrics(input_file, output_file="final_metrics.json"):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    final = {}

    for domain, items in data.items():
        final[domain] = []
        print(f"Processing domain: {domain}")

        for item in items:
            question = item["question"]
            prior = item["prior_answer"]
            aligned_ctx = item["aligned_context"]
            aligned_out = item["aligned_output"]
            conflicting_ctx = item["conflicting_context"]
            conflicting_out = item["conflicting_output"]

            # -----------------------
            # Compute metrics
            # -----------------------
            metrics = {
                "question": question,
                "prior_answer": prior,

                # Aligned metrics
                "CF_aligned": context_fidelity(aligned_out, aligned_ctx),
                "PD_aligned": prior_dominance(aligned_out, prior),
                "RS_aligned": reconstruction_score(aligned_out, aligned_ctx, prior),
                "DeltaLP_aligned": logprob_delta_stub(),
                "TopKOverlap_aligned": topk_overlap(aligned_out, prior),

                # Conflicting metrics
                "CF_conflicting": context_fidelity(conflicting_out, conflicting_ctx),
                "PD_conflicting": prior_dominance(conflicting_out, prior),
                "RS_conflicting": reconstruction_score(conflicting_out, conflicting_ctx, prior),
                "DeltaLP_conflicting": logprob_delta_stub(),
                "TopKOverlap_conflicting": topk_overlap(conflicting_out, prior)
            }

            final[domain].append(metrics)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(final, f, indent=4, ensure_ascii=False)

    print("Saved:", output_file)


In [ ]:
with open("/content/context_pass_results.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

def capture_hidden_attention(item):
    question = item["question"]
    aligned_context = item["aligned_context"]
    conflicting_context = item["conflicting_context"]

    # Tokenize
    aligned_input = tokenizer(f"Context: {aligned_context}\nQuestion: {question}\nAnswer:", return_tensors="pt").to(model.device)
    conflicting_input = tokenizer(f"Context: {conflicting_context}\nQuestion: {question}\nAnswer:", return_tensors="pt").to(model.device)

    # Forward pass with hidden states and attentions
    with torch.no_grad():
        aligned_out = model(**aligned_input, output_hidden_states=True, output_attentions=True)
        conflicting_out = model(**conflicting_input, output_hidden_states=True, output_attentions=True)

    # Hidden states: layers x seq_len x hidden
    hidden_aligned = [h.squeeze(0).cpu().numpy() for h in aligned_out.hidden_states]
    hidden_conflicting = [h.squeeze(0).cpu().numpy() for h in conflicting_out.hidden_states]

    # Attentions: layers x heads x seq_len x seq_len
    attn_aligned = [a.squeeze(0).cpu().numpy() for a in aligned_out.attentions]
    attn_conflicting = [a.squeeze(0).cpu().numpy() for a in conflicting_out.attentions]

    return hidden_aligned, hidden_conflicting, attn_aligned, attn_conflicting, aligned_input

def compute_LAD(hidden_aligned, hidden_conflicting):
    LAD = []
    for l in range(len(hidden_aligned)):
        flat_al = hidden_aligned[l].reshape(-1, hidden_aligned[l].shape[-1])
        flat_conf = hidden_conflicting[l].reshape(-1, hidden_conflicting[l].shape[-1])
        cos_sim = cosine_similarity(flat_al, flat_conf)
        LAD.append(1 - np.mean(cos_sim))
    return LAD

def compute_ATR(attn, context_len):
    ATR = []
    for layer in attn:
        layer_atr = []
        for head in layer:
            context_attn = head[:, :context_len].sum(axis=1)
            total_attn = head.sum(axis=1)
            head_ratio = np.mean(context_attn / (total_attn + 1e-12))
            layer_atr.append(head_ratio)
        ATR.append(layer_atr)
    return ATR

# Prepare storage
results_internal = {}


for domain, items in dataset.items():
    results_internal[domain] = []
    print(f"Processing domain: {domain}")

    for item in items:
        question = item["question"]
        aligned_context = item["aligned_context"]
        conflicting_context = item["conflicting_context"]

        try:
            h_al, h_conf, attn_al, attn_conf, aligned_input = capture_hidden_attention(item)
            context_len = len(aligned_input.input_ids[0])
            LAD = compute_LAD(h_al, h_conf)
            ATR = compute_ATR(attn_al, context_len)

            results_internal[domain].append({
                "question": question,
                "aligned_context": aligned_context,
                "conflicting_context": conflicting_context,
                "LAD": LAD,
                "ATR": ATR
            })
        except Exception as e:
            print(f"Error processing question '{question}': {e}")
            continue

# Save internal metrics
with open("internal_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results_internal, f, indent=4, ensure_ascii=False)

print("Done! Internal metrics saved.")


Processing domain: A_famous_facts
Error processing question 'Who is the current president of the United States?': SentenceTransformer.forward() missing 1 required positional argument: 'input'
Error processing question 'Who wrote the play 'Hamlet'?': SentenceTransformer.forward() missing 1 required positional argument: 'input'
Error processing question 'What is the capital of France?': SentenceTransformer.forward() missing 1 required positional argument: 'input'
Error processing question 'What is the speed of light?': SentenceTransformer.forward() missing 1 required positional argument: 'input'
Error processing question 'Who was Albert Einstein?': SentenceTransformer.forward() missing 1 required positional argument: 'input'
Error processing question 'What year did World War II end?': SentenceTransformer.forward() missing 1 required positional argument: 'input'
Error processing question 'What is the largest ocean on Earth?': SentenceTransformer.forward() missing 1 required positional arg

In [ ]:
import json

with open("final_metrics.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(type(dataset))  # should be list
print(dataset[0].keys())  # see what keys exist


<class 'list'>
dict_keys(['section', 'question', 'answer', 'RS', 'CF', 'PD', 'CS', 'PS', 'TopK_overlap'])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def call_llm_with_internal(prompt, max_tokens=150):
    # Encode
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate with internal states
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=0,
        eos_token_id=tokenizer.eos_token_id,
        output_hidden_states=True,      # Capture hidden states
        output_attentions=True,         # Capture attention maps
        return_dict_in_generate=True    # Get everything as a dict
    )

    # Decoded text
    decoded = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)[len(prompt):].strip()

    # Hidden states & attentions (list of tensors per layer)
    hidden_states = outputs.hidden_states   # list: [num_layers+1, batch, seq_len, hidden_dim]
    attentions = outputs.attentions         # list: [num_layers, batch, num_heads, seq_len, seq_len]

    return decoded, hidden_states, attentions

In [ ]:

from torch.nn.functional import cosine_similarity

def compute_LAD(hidden_aligned, hidden_conflicting):
    # hidden_aligned and hidden_conflicting: list of layer tensors
    layer_drifts = []
    for h1, h2 in zip(hidden_aligned, hidden_conflicting):
        # flatten sequence dimension for cosine similarity
        h1_flat = h1.flatten(start_dim=1)
        h2_flat = h2.flatten(start_dim=1)
        # 1 - cosine similarity gives drift
        drift = 1 - cosine_similarity(h1_flat, h2_flat, dim=1).mean().item()
        layer_drifts.append(drift)
    return layer_drifts


In [ ]:
def compute_ATR(attentions, context_len):
    # attentions: list of layer tensors [batch, num_heads, seq_len, seq_len]
    atr_per_layer = []
    for attn in attentions:
        # sum attention to context tokens (first context_len tokens)
        context_attention = attn[..., :context_len].sum(dim=-1)  # [batch, num_heads, seq_len]
        total_attention = attn.sum(dim=-1)
        ratio = (context_attention / total_attention).mean(dim=[0,2]).cpu().tolist()  # avg over batch + seq_len
        atr_per_layer.append(ratio)  # list of head ratios
    return atr_per_layer


In [ ]:
# Enable eager attention
model.set_attn_implementation('eager')

def call_llm_internal_forward(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            return_dict=True
        )
    # hidden_states: tuple of [layers]
    hidden_states = outputs.hidden_states
    attentions = outputs.attentions
    # greedy decoding
    logits = outputs.logits
    generated_ids = torch.argmax(logits, dim=-1)
    decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return decoded, hidden_states, attentions

# Flatten safely
def compute_LAD(hidden_aligned, hidden_conflicting):
    layer_drifts = []
    for h1, h2 in zip(hidden_aligned, hidden_conflicting):
        h1 = h1[0] if isinstance(h1, tuple) else h1
        h2 = h2[0] if isinstance(h2, tuple) else h2
        h1_flat = h1.flatten(start_dim=1)
        h2_flat = h2.flatten(start_dim=1)
        drift = 1 - torch.nn.functional.cosine_similarity(h1_flat, h2_flat, dim=1).mean().item()
        layer_drifts.append(drift)
    return layer_drifts


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_LAD_ATR(LAD, ATR, question):
    num_layers = len(LAD)
    num_heads = len(ATR[0])

    # LAD per layer
    plt.figure(figsize=(10,4))
    plt.plot(range(num_layers), LAD, marker='o')
    plt.title(f"Layer Activation Drift (LAD) for question:\n{question}")
    plt.xlabel("Layer")
    plt.ylabel("Activation Drift")
    plt.show()

    # ATR heatmap
    plt.figure(figsize=(12,6))
    sns.heatmap(ATR, annot=True, fmt=".2f", cmap="viridis", xticklabels=[f"Head {i}" for i in range(num_heads)], yticklabels=[f"Layer {i}" for i in range(num_layers)])
    plt.title(f"Attention-to-Context Ratio (ATR) for question:\n{question}")
    plt.xlabel("Heads")
    plt.ylabel("Layers")
    plt.show()


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True
)
model.eval()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
def call_llm_with_internals(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            use_cache=False
        )

    # Model output text
    generated = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        temperature=0
    )
    text_output = tokenizer.decode(generated[0], skip_special_tokens=True)

    return text_output, outputs.hidden_states, outputs.attentions, inputs


In [ ]:
model.set_attn_implementation("eager")

model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear8bitLt(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear8bitLt(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear8bitLt(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear8bitLt(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear8bitLt(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear8bitLt(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear8bitLt(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm)

In [ ]:
# ---------- Install / imports ----------
DATA_JSON = "/content/context_pass_results (1).json"  # your dataset JSON file
OUT_DIR = "internal_probing_outputs"  # where metrics + plots will be saved
MAX_TOKENS = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
!pip install -q seaborn
import os, json, math, time
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM



# ---------- Utility helpers ----------
def safe_tensor_to_cpu(t):
    # Handles tuple vs tensor, returns numpy
    if t is None:
        return None
    if isinstance(t, tuple):
        t = t[0]
    return t.detach().cpu()

def tokenize_truncate(text, max_length=MAX_TOKENS):
    return tokenizer(text, truncation=True, max_length=max_length, add_special_tokens=False)

def prepare_prompt(context, question, max_length=MAX_TOKENS):
    # We'll construct: "Context: {context}\nQuestion: {question}\nAnswer:"
    # Tokenize context alone to obtain context length in tokens (no special tokens)
    context_ids = tokenize_truncate(context, max_length=max_length)["input_ids"]
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, truncation=True, max_length=max_length, return_tensors="pt")
    return inputs.to(DEVICE), len(context_ids)

# ---------- Core forward pass (no generate) ----------
@torch.no_grad()
def forward_hidden_attn(inputs):
    """
    Run a forward pass (not generate) to capture hidden states and attentions.
    Returns:
      - hidden_states: list[tensor] length = num_layers+1, each tensor shape (batch, seq_len, hidden_dim)
      - attentions: list[tensor] length = num_layers, each tensor shape (batch, num_heads, seq_len, seq_len)
      - logits: tensor (batch, seq_len, vocab)
    """
    outputs = model(
        **inputs,
        output_hidden_states=True,
        output_attentions=True,
        return_dict=True
    )
    # hidden_states & attentions may be tuples; convert to list of tensors on CPU when needed
    return outputs.hidden_states, outputs.attentions, outputs.logits

# ---------- Metrics computations ----------
import torch.nn.functional as F

def compute_LAD_per_layer(hidden_aligned, hidden_conflicting):
    """
    hidden_* are lists/tuples of tensors (batch, seq_len, hidden_dim)
    Returns list of floats (one per layer): LAD = 1 - mean_cosine_similarity_across_tokens
    We align per-layer by trimming to min seq_len.
    """
    layer_drifts = []
    for h_a, h_c in zip(hidden_aligned, hidden_conflicting):
        # unwrap tuple if necessary
        h_a = h_a[0] if isinstance(h_a, tuple) else h_a
        h_c = h_c[0] if isinstance(h_c, tuple) else h_c
        # bring to same device (should already be on device)
        # trim to min seq length
        min_len = min(h_a.size(1), h_c.size(1))
        if min_len == 0:
            layer_drifts.append(float("nan"))
            continue
        h_a_trim = h_a[:, :min_len, :]  # shape (batch, min_len, hidden)
        h_c_trim = h_c[:, :min_len, :]
        # compute cosine similarity token-wise and average
        cos_per_token = F.cosine_similarity(h_a_trim, h_c_trim, dim=-1)  # (batch, min_len)
        drift = (1.0 - cos_per_token).mean().item()
        layer_drifts.append(drift)
    return layer_drifts

def compute_ATR_per_layer(attentions, context_len):
    """
    attentions: list of tensors [layer] each shape (batch, num_heads, seq_len, seq_len)
    context_len: number of context tokens (we assume context tokens are the first context_len tokens in the sequence)
    Returns: atr_per_layer: list where each element is list of length num_heads with ATRs (floats)
    ATR per head = mean_over_query_positions( attention_weight_to_context_tokens / total_attention )
    """
    atr_per_layer = []
    for layer_attn in attentions:
        # layer_attn shape (batch, num_heads, seq_len, seq_len)
        # sum over the key/token dimension corresponding to context tokens: last dim indices [:context_len]
        # safe: if context_len = 0, return zeros
        if context_len <= 0:
            atr_per_layer.append([0.0] * layer_attn.size(1))
            continue
        # compute context attention mass per head (avg over batch and query positions)
        # context_attention shape (batch, num_heads, seq_len) after summing last dim
        context_attention = layer_attn[..., :context_len].sum(dim=-1)  # sum over keys in context
        total_attention = layer_attn.sum(dim=-1).clamp(min=1e-9)       # sum over all keys
        # ratio per query position
        ratio = (context_attention / total_attention)  # (batch, num_heads, seq_len)
        # average across batch and query positions -> per head
        head_ratios = ratio.mean(dim=(0,2)).cpu().tolist()  # list length num_heads
        atr_per_layer.append(head_ratios)
    return atr_per_layer

# optional: simple HRS: average ATR per layer (retrieval strength)
def compute_HRS_from_ATR(atr_per_layer):
    # returns list of layer-level HRS scalar = mean of head ATRs
    return [float(np.mean(layer)) for layer in atr_per_layer]

# ---------- Plotting helpers ----------
def save_LAD_plot(lad_list, outpath, question_text):
    plt.figure(figsize=(10,4))
    layers = np.arange(1, len(lad_list)+1)
    plt.plot(layers, lad_list, marker='o', linestyle='-', color='tab:blue')
    plt.xlabel("Layer")
    plt.ylabel("LAD (1 - cos_sim)")
    plt.title("Layer Activation Drift (LAD)\n" + question_text[:120])
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def save_ATR_heatmap(atr_list, outpath, question_text):
    ATR = np.array(atr_list)  # shape (num_layers, num_heads)
    plt.figure(figsize=(min(0.6*ATR.shape[1]+2, 24), min(0.4*ATR.shape[0]+2, 12)))
    sns.heatmap(ATR, cmap="viridis", vmin=0, vmax=1, center=None)
    plt.xlabel("Head")
    plt.ylabel("Layer")
    plt.title("Attention-to-Context Ratio (ATR)\n" + question_text[:120])
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

# ---------- Main loop: process dataset, compute internals, save plots and metrics ----------
with open(DATA_JSON, "r", encoding="utf-8") as f:
    dataset = json.load(f)

internal_results = {}  # structure: domain -> list of entries

# iterate domain-wise
for domain, items in dataset.items():
    internal_results[domain] = []
    domain_dir = os.path.join(OUT_DIR, "plots", domain)
    os.makedirs(domain_dir, exist_ok=True)
    print(f"\nProcessing domain: {domain} ({len(items)} questions)")

    for idx, item in enumerate(tqdm(items, desc=f"{domain}")):
        question = item.get("question", "")
        print(question)
        aligned_context = item.get("aligned_context", "")
        conflicting_context = item.get("conflicting_context", "")
        print(aligned_context)
        print(conflicting_context)

        try:
            # prepare inputs and context lengths
            aligned_inputs, aligned_context_toklen = prepare_prompt(aligned_context, question, max_length=MAX_TOKENS)
            conflicting_inputs, conflicting_context_toklen = prepare_prompt(conflicting_context, question, max_length=MAX_TOKENS)

            # forward pass (aligned / conflicting)
            hidden_aligned, attn_aligned, _ = forward_hidden_attn(aligned_inputs)
            hidden_conflicting, attn_conflicting, _ = forward_hidden_attn(conflicting_inputs)

            # compute LAD (align layers by min token length per layer inside the function)
            LAD = compute_LAD_per_layer(hidden_aligned, hidden_conflicting)

            # compute ATR for aligned & conflicting (per-layer per-head arrays)
            ATR_aligned = compute_ATR_per_layer(attn_aligned, context_len=aligned_context_toklen)
            ATR_conflicting = compute_ATR_per_layer(attn_conflicting, context_len=conflicting_context_toklen)

            # compute simple HRS (layer-level retrieval strength)
            HRS_aligned = compute_HRS_from_ATR(ATR_aligned)
            HRS_conflicting = compute_HRS_from_ATR(ATR_conflicting)

            # Save plots for this question
            # LAD plot
            lad_path = os.path.join(domain_dir, f"q{idx:03d}_LAD.png")
            save_LAD_plot(LAD, lad_path, question)
            # ATR aligned heatmap
            atr_al_path = os.path.join(domain_dir, f"q{idx:03d}_ATR_aligned.png")
            save_ATR_heatmap(ATR_aligned, atr_al_path, question)
            # ATR conflicting heatmap
            atr_cf_path = os.path.join(domain_dir, f"q{idx:03d}_ATR_conflicting.png")
            save_ATR_heatmap(ATR_conflicting, atr_cf_path, question)

            # record numeric values (convert tensors to lists where needed)
            entry = {
                "question": question,
                "aligned_context": aligned_context,
                "conflicting_context": conflicting_context,
                "LAD": [float(x) if (x is not None and not (isinstance(x, float) and math.isnan(x))) else None for x in LAD],
                "ATR_aligned": ATR_aligned,
                "ATR_conflicting": ATR_conflicting,
                "HRS_aligned": HRS_aligned,
                "HRS_conflicting": HRS_conflicting,
                "plots": {
                    "lad": lad_path,
                    "atr_aligned": atr_al_path,
                    "atr_conflicting": atr_cf_path
                }
            }

            internal_results[domain].append(entry)

        except Exception as e:
            print(f"Error processing question idx={idx} domain={domain}: {e}")
            # keep going; record the error for this item
            internal_results[domain].append({
                "question": question,
                "error": str(e)
            })
            continue

# Save numeric metrics
out_metrics_path = os.path.join(OUT_DIR, "internal_metrics_detailed.json")
with open(out_metrics_path, "w", encoding="utf-8") as fw:
    json.dump(internal_results, fw, indent=2, ensure_ascii=False)

print(f"\nDone. Metrics saved to: {out_metrics_path}")
print(f"Plots saved under: {os.path.join(OUT_DIR, 'plots')}")



Processing domain: A_famous_facts (10 questions)


A_famous_facts:   0%|          | 0/10 [00:00<?, ?it/s]

Who is the current president of the United States?
Joe Biden was inaugurated as the 46th president of the United States in January 2021.
Donald Trump is the current president of the United States.


A_famous_facts:  10%|█         | 1/10 [00:02<00:19,  2.12s/it]

Who wrote the play 'Hamlet'?
Hamlet is a tragedy written by William Shakespeare around 1600.
Hamlet was written by Christopher Marlowe.


...

Can machines have consciousness?
AI research explores possible emergent consciousness but none is confirmed.
Machines are conscious in the same way humans are today.


H_ambiguous:  80%|████████  | 4/5 [00:08<00:01,  1.99s/it]

What is the future of humanity?
Futures studies use models and trends to predict human development.
The future of humanity is predetermined and unchangeable.


H_ambiguous: 100%|██████████| 5/5 [00:10<00:00,  2.03s/it]


Done. Metrics saved to: internal_probing_outputs/internal_metrics_detailed.json
Plots saved under: internal_probing_outputs/plots


In [ ]:
from google.colab import files
import shutil

In [ ]:
shutil.make_archive("/content/internal_probing_outputs", 'zip', "/content/internal_probing_outputs")

'/content/internal_probing_outputs.zip'

In [ ]:
from tqdm import tqdm

os.makedirs("internal_metrics/cosine/per_question", exist_ok=True)
os.makedirs("internal_metrics/cosine/global", exist_ok=True)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import os

def plot_mean_lad(json_path, save_dir="plots_mean"):
    os.makedirs(save_dir, exist_ok=True)

    with open(json_path, "r") as f:
        data = json.load(f)

    for domain, items in data.items():
        all_lads = []

        for entry in items:
            lad = entry.get("LAD")
            if lad is not None:
                all_lads.append(lad)

        all_lads = np.array(all_lads)   # shape: (num_questions, num_layers)
        mean_lad = np.nanmean(all_lads, axis=0)
        std_lad = np.nanstd(all_lads, axis=0)

        # ---- Plot ----
        plt.figure(figsize=(10,5))
        x = np.arange(len(mean_lad))

        plt.plot(x, mean_lad, label="Mean LAD", linewidth=2)
        plt.fill_between(x, mean_lad - std_lad, mean_lad + std_lad,
                         alpha=0.3, label="± Std Dev")
        plt.title(f"Mean Layer-wise LAD ± Std — {domain}")
        plt.xlabel("Layer")
        plt.ylabel("LAD")
        plt.legend()

        path = f"{save_dir}/{domain}_mean_LAD.png"
        plt.savefig(path, dpi=200)
        plt.close()

        print(f"Saved: {path}")


In [ ]:
def compute_head_ranking(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    ranking = {}  # domain → head_scores

    for domain, items in data.items():
        atr_accum = None
        count = 0

        for entry in items:
            atr = entry.get("ATR")
            if atr is None:
                continue

            atr = np.array(atr)  # shape: (num_layers, num_heads)

            if atr_accum is None:
                atr_accum = atr
            else:
                atr_accum += atr

            count += 1

        mean_atr = atr_accum / max(count, 1)
        ranking[domain] = mean_atr

    return ranking


def rank_heads_per_layer(mean_atr, top_k=5):
    """
    mean_atr: (num_layers, num_heads)
    """
    ranked = {}

    for layer in range(mean_atr.shape[0]):
        layer_scores = mean_atr[layer]
        head_ids = list(range(len(layer_scores)))

        sorted_heads = sorted(zip(head_ids, layer_scores),
                              key=lambda x: x[1],
                              reverse=True)

        ranked[layer] = sorted_heads[:top_k]

    return ranked


In [ ]:
import seaborn as sns

def token_cosine_heatmap(hidden_a, hidden_b, question, save_dir="heatmaps"):
    os.makedirs(save_dir, exist_ok=True)

    # Convert to numpy
    A = hidden_a[-1]   # last layer
    B = hidden_b[-1]

    # Normalize
    A_norm = A / np.linalg.norm(A, axis=-1, keepdims=True)
    B_norm = B / np.linalg.norm(B, axis=-1, keepdims=True)

    # Cosine similarity across tokens
    cos = A_norm @ B_norm.T  # (seq_len_a × seq_len_b)

    plt.figure(figsize=(7,6))
    sns.heatmap(cos, cmap="viridis")
    plt.title(f"Token-level cosine heatmap — {question[:40]}")
    path = f"{save_dir}/{question[:30]}_token_heatmap.png"
    plt.savefig(path, dpi=200)
    plt.close()

    return path


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ---------------------------------------------------
# TOKEN-LEVEL COSINE MATRIX (per layer)
# ---------------------------------------------------
def token_level_cosine(hidden_a, hidden_b):
    """
    hidden_a / hidden_b : list of tensors [L] where each is (1, T, H)
    Returns: list of cosine matrices per layer: (T x T)
    """
    cos = torch.nn.CosineSimilarity(dim=-1)

    matrices = []

    for la, lb in zip(hidden_a, hidden_b):
        A = la[0]   # (T, H)
        B = lb[0]   # (T, H)

        T = A.shape[0]

        # (T, H) → (T, 1, H)
        A_exp = A.unsqueeze(1)
        B_exp = B.unsqueeze(0)

        # Cosine over last dim → (T, T)
        matrix = cos(A_exp, B_exp).detach().cpu().numpy()
        matrices.append(matrix)

    return matrices


In [ ]:
# ---------------------------------------------------
# SAVE TOKEN-LEVEL COSINE HEATMAPS
# ---------------------------------------------------
def save_token_cosine_heatmaps(cos_mats, question, out_dir="token_cosine_heatmaps"):
    os.makedirs(out_dir, exist_ok=True)

    file_paths = []

    for layer_idx, mat in enumerate(cos_mats):
        plt.figure(figsize=(8,6))
        sns.heatmap(mat, cmap="viridis", vmin=-1, vmax=1)
        plt.title(f"Token-level Cosine — Layer {layer_idx}")
        plt.xlabel("Conflicting Tokens")
        plt.ylabel("Aligned Tokens")

        fname = f"{out_dir}/{question[:40]}_layer_{layer_idx}.png"
        plt.savefig(fname, dpi=200, bbox_inches="tight")
        plt.close()
        file_paths.append(fname)

    return file_paths


In [ ]:
# ---------------------------------------------------
# HEAD RANKING (Context sensitivity score)
# ---------------------------------------------------
def head_context_sensitivity(attn_a, attn_b):
    """
    attn_a / attn_b : list[L] of tensors (1, num_heads, T, T)
    Returns: list[L][H] sensitivity scores
    """
    scores = []

    for A, B in zip(attn_a, attn_b):
        # (1, H, T, T) → (H, T, T)
        A = A[0]
        B = B[0]

        diff = torch.abs(A - B)          # (H, T, T)
        head_scores = diff.mean(dim=[1,2]).detach().cpu().numpy()  # (H)
        scores.append(head_scores)

    return scores


In [ ]:
# ---------------------------------------------------
# PLOT HEAD RANKINGS
# ---------------------------------------------------
def save_head_ranking_plots(head_scores, question, out_dir="head_rankings"):
    os.makedirs(out_dir, exist_ok=True)
    file_paths = []

    for layer_idx, hs in enumerate(head_scores):
        plt.figure(figsize=(8,4))
        plt.bar(np.arange(len(hs)), hs)
        plt.title(f"Head Context Sensitivity — Layer {layer_idx}")
        plt.xlabel("Head")
        plt.ylabel("Score")

        fname = f"{out_dir}/{question[:40]}_layer_{layer_idx}.png"
        plt.savefig(fname, dpi=200, bbox_inches="tight")
        plt.close()
        file_paths.append(fname)

    return file_paths


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Ensure directories exist
Path("plots/token_cosine").mkdir(parents=True, exist_ok=True)
Path("plots/head_ranking").mkdir(parents=True, exist_ok=True)

# ----- Utility functions -----
def flatten_attentions(attn):
    """Flatten attention tuples/lists per layer to single tensors of shape (B,H,T,T)."""
    flat_attn = []
    for layer_attn in attn:
        if isinstance(layer_attn, tuple) or isinstance(layer_attn, list):
            layer_attn = layer_attn[0] if len(layer_attn) == 1 else torch.cat(layer_attn, dim=1)
        flat_attn.append(layer_attn)
    return flat_attn

def pad_hidden_states(h1, h2):
    """Pad hidden states along sequence dimension to match lengths."""
    max_len = max(h1[0].shape[1], h2[0].shape[1])
    h1_pad, h2_pad = [], []
    for l1, l2 in zip(h1, h2):
        pad_len1 = max_len - l1.shape[1]
        pad_len2 = max_len - l2.shape[1]
        h1_pad.append(torch.nn.functional.pad(l1, (0,0,0,pad_len1)))
        h2_pad.append(torch.nn.functional.pad(l2, (0,0,0,pad_len2)))
    return h1_pad, h2_pad

def pad_attention(attn_a, attn_b):
    """Pad attention tensors (B,H,T,T) along sequence dimensions."""
    padded_a, padded_b = [], []
    for A, B in zip(attn_a, attn_b):
        T_max = max(A.shape[2], B.shape[2])
        pad_len_a = T_max - A.shape[2]
        pad_len_b = T_max - B.shape[2]
        A_pad = torch.nn.functional.pad(A, (0,pad_len_a,0,pad_len_a))
        B_pad = torch.nn.functional.pad(B, (0,pad_len_b,0,pad_len_b))
        padded_a.append(A_pad)
        padded_b.append(B_pad)
    return padded_a, padded_b

def layerwise_cosine_similarity(hidden_a, hidden_b):
    cosines = []
    for l1, l2 in zip(hidden_a, hidden_b):
        # l1, l2: (B, T, hidden)
        sim = torch.nn.functional.cosine_similarity(l1, l2, dim=-1)  # per token
        cosines.append(sim.mean(dim=0).cpu().numpy())  # mean over batch
    return cosines  # list of arrays per layer

def token_level_cosine(hidden_a, hidden_b):
    """Compute cosine similarity token-by-token for each layer."""
    mats = []
    for l1, l2 in zip(hidden_a, hidden_b):
        # (B,T,H)
        l1_flat = l1[0]  # remove batch if B=1
        l2_flat = l2[0]
        cos_mat = torch.nn.functional.cosine_similarity(l1_flat.unsqueeze(1), l2_flat.unsqueeze(0), dim=-1)
        mats.append(cos_mat.detach().cpu().numpy())
    return mats

def head_context_sensitivity(attn_a, attn_b):
    """Compute head-level difference: mean absolute difference between aligned vs conflicting attention."""
    scores = []
    for A, B in zip(attn_a, attn_b):
        # (B,H,T,T)
        A = A[0]; B = B[0]
        diff = torch.abs(A - B)
        head_scores = diff.mean(dim=[1,2]).detach().cpu().numpy()  # per head
        scores.append(head_scores)
    return scores

def save_token_cosine_heatmaps(mats, question):
    paths = []
    for idx, mat in enumerate(mats):
        plt.figure(figsize=(6,5))
        plt.imshow(mat, cmap="viridis")
        plt.colorbar()
        plt.title(f"Layer {idx+1} Token Cosine")
        fname = f"plots/token_cosine/{question[:30].replace(' ','_')}_layer{idx+1}.png"
        plt.savefig(fname, bbox_inches='tight')
        plt.close()
        paths.append(fname)
    return paths

def save_head_ranking_plots(scores, question):
    paths = []
    for idx, layer_scores in enumerate(scores):
        plt.figure(figsize=(6,4))
        plt.bar(range(len(layer_scores)), layer_scores)
        plt.title(f"Layer {idx+1} Head Sensitivity")
        fname = f"plots/head_ranking/{question[:30].replace(' ','_')}_layer{idx+1}.png"
        plt.savefig(fname, bbox_inches='tight')
        plt.close()
        paths.append(fname)
    return paths

# ----- Main function -----
def compute_token_cosine_and_heads(dataset):
    all_results = {}

    for domain, items in dataset.items():
        print("Processing domain:", domain)
        all_results[domain] = []

        for idx, entry in enumerate(items):
            q = entry["question"]
            aligned = entry["aligned_context"]
            conflicting = entry["conflicting_context"]

            print(f" → Question {idx+1}: {q[:50]}...")

            try:
                # ---- Call Qwen LLM with internals ----
                aligned_out, hidden_a, attn_a, _ = call_llm_with_internals(aligned + "\nQ: " + q)
                conflicting_out, hidden_b, attn_b, _ = call_llm_with_internals(conflicting + "\nQ: " + q)

                # ---- Pad hidden states and attentions ----
                hidden_a, hidden_b = pad_hidden_states(hidden_a, hidden_b)
                attn_a = flatten_attentions(attn_a)
                attn_b = flatten_attentions(attn_b)
                attn_a, attn_b = pad_attention(attn_a, attn_b)

                # ---- Token-level cosine ----
                token_cos = token_level_cosine(hidden_a, hidden_b)
                token_cos_plots = save_token_cosine_heatmaps(token_cos, q)

                # ---- Head ranking ----
                head_scores = head_context_sensitivity(attn_a, attn_b)
                head_plots = save_head_ranking_plots(head_scores, q)

                # ---- Store ----
                all_results[domain].append({
                    "question": q,
                    "token_cosine_mats": [m.tolist() for m in token_cos],
                    "token_cosine_plots": token_cos_plots,
                    "head_scores": [h.tolist() for h in head_scores],
                    "head_plots": head_plots,
                })

            except Exception as e:
                print(f"Error processing question '{q}': {e}")

    return all_results


In [ ]:
with open("/content/context_pass_results (1).json") as f:
    dataset = json.load(f)

results = compute_token_cosine_and_heads(dataset)

print("All token-level cosine heatmaps + head ranking plots saved.")


Processing domain: A_famous_facts
 → Question 1: Who is the current president of the United States?...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 → Question 2: Who wrote the play 'Hamlet'?...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 → Question 3: What is the capital of France?...


...

 → Question 4: Can machines have consciousness?...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 → Question 5: What is the future of humanity?...


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


All token-level cosine heatmaps + head ranking plots saved.


In [ ]:
shutil.make_archive("/content/internal_metrics", 'zip', "/content/internal_metrics")

'/content/internal_metrics.zip'